<a href="https://colab.research.google.com/github/sbleeedu1-ai/python_CLI_board_example_26_08/blob/main/%ED%8C%8C%EC%9D%B4%EC%8D%AC_CLI_board.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 게시판
from datetime import datetime
from pytz import timezone
from dataclasses import dataclass
import getpass    # 비밀번호 보이지 않게

@dataclass
class Article:
  id: int
  regDate: str
  updateDate: str
  title: str
  content: str
  writeId: str

@dataclass
class Member:
  id: int
  regDate: str
  updateDate: str
  loginId : str
  loginPw: str
  name: str


def now():
  return datetime.now(timezone('Asia/Seoul')).strftime("%Y-%m-%d %H:%M:%S")

def parse_id(cmd):
  cmd_bits = cmd.split(" ")

  if len(cmd_bits) < 3:
    print("명령어를 다시 입력해주세요 (글 번호 없음)")
    return None

  if not cmd_bits[2].isdigit():
    print("명령어를 다시 입력해주세요 (번호대신 문자입력)")
    return None

  return int(cmd_bits[2])

def find_article(articles,article_id):
  for article in articles:
    if article.id == article_id:
      return article
  return None

def format_date(regDate):
  today = now().split(" ")[0]
  write_date = regDate.split(" ")[0]
  write_time = regDate.split(" ")[1]
  if today == write_date:
    return write_time
  else:
    return write_date

def make_article_TestData():
  return [
      Article(1, "2025-12-12 12:12:12", "2025-12-12 12:12:12", "제목1", "내용1","test1"),
      Article(2, now(), now(), "제목2", "내용2","test1"),
      Article(3, now(), now(), "제목3", "내용3","test2")
          ]

def make_member_TestData():
  return [
      Member(1, now(), now(), "test1", "test1", "회원1"),
      Member(2, now(), now(), "test2", "test2", "회원2")
          ]

def find_member(members,loginId):
  for member in members:
    if member.loginId == loginId:
      print("아이디가 이미 존재합니다.")
      return loginId
  return None
# find_article 함수와 동일하게 리스트에서 꺼내 하나씩 대조

def pw_check(members,loginId,loginPw):
  for member in members:
    if member.loginId == loginId:
      if member.loginPw == loginPw:
        return member
  return None


print("== CLI 게시판 실행 ==")
article_num = 3
articles = make_article_TestData()

member_num = 2
members = make_member_TestData()

loginMember = None      # 로그인 상태 저장

while True:
  user_cmd = input("명령어 ) ").strip()

  if user_cmd == 'exit':
    break

  # 회원 Member
  # 회원 가입
  # member join

  elif  user_cmd == 'member join':
    if loginMember is not None:
      print("이미 로그인한 상태입니다.")
      continue

    member_num += 1

    while True:
      loginId = input("아이디 : ")
      isDup = find_member(members,loginId)
      if isDup is not None:
        continue
      break

    while True:
      loginPw = getpass.getpass("암호 : ")
      loginPwConfirm = getpass.getpass("암호 확인 : ")
      if loginPw != loginPwConfirm:
        print("암호가 일치하지 않습니다.")
        continue
      break

    name = input("이름 : ")
    member = Member(member_num,now(),now(),loginId,loginPw,name)
    members.append(member)
    print(f"{member_num}번째 회원 가입을 축하합니다.")

  # 로그인
  # member login

  elif  user_cmd == 'member login':
    if loginMember is not None:
      print("이미 로그인한 상태입니다.")
      continue

    login_id = input("아이디 : ")
    login_pw = getpass.getpass("암호 : ")
    member = pw_check(members, login_id, login_pw)

    if member is None:
      print("로그인 실패")
      continue

    loginMember = member
    print("로그인 성공")

  # 로그아웃
  # member logout

  elif user_cmd == 'member logout':
    if loginMember is None:
      print("로그인 하지 않았습니다.")
      continue

    loginMember = None
    print("로그아웃 되었습니다.")

  elif user_cmd == 'article write':
    if loginMember is None:
      print("로그인이 필요합니다.")
      continue

    article_num += 1
    title = input("제목 : ")
    content = input("내용 : ")
    article = Article(article_num,now(),now(),title,content,loginMember.loginId)
    articles.append(article)
    print(f"{article_num}번 글이 생성되었습니다.")

  elif user_cmd.startswith('article delete'):
    if not loginMember:
      print("로그인이 필요합니다.")
      continue

    deletedId = parse_id(user_cmd)
    if deletedId is None:
      continue

    article = find_article(articles, deletedId)
    if article is None:
      print(f"{deletedId}번 글은 존재하지 않습니다.")
      continue

    if article.writeId != loginMember.loginId:
      print("작성자가 아닙니다.")
      continue

    articles.remove(article)
    print(f"{deletedId}번 글이 삭제 되었습니다.")

  elif user_cmd.startswith('article edit'):
    if not loginMember:
      print("로그인이 필요합니다.")
      continue

    editedId = parse_id(user_cmd)
    if editedId is None:
      continue

    article = find_article(articles, editedId)
    if article is None:
      print(f"{editedId}번 글은 존재하지 않습니다.")
      continue

    if article.writeId != loginMember.loginId:
      print("작성자가 아닙니다.")
      continue

    print(f"기존 제목 : {article.title}")
    print(f"기존 내용 : {article.content}")
    article.title = input("새 제목 : ")
    article.content = input("새 내용 : ")
    article.updateDate = now()
    print(f"{editedId}번 글이 수정 되었습니다.")

  elif user_cmd.startswith('article detail'):
    detailId = parse_id(user_cmd)
    if detailId is None:
      continue

    article = find_article(articles, detailId)
    if article is None:
      print(f"{detailId}번 글은 존재하지 않습니다.")
      continue

    print(f"번호 : {article.id}")
    print(f"작성 날짜 : {article.regDate}")
    print(f"수정 날짜 : {article.updateDate}")
    print(f"작성자 : {article.writeId}")
    print(f"제목 : {article.title}")
    print(f"내용 : {article.content}")

  elif user_cmd == 'article list':
    if not articles:
      print("작성한 글이 없습니다.")
    else:
      print("========================================================")
      print("번호".ljust(5),end='/')
      print("    제목".ljust(10),end='/')
      print("    내용".ljust(10),end='/')
      print("    작성자".ljust(10),end='/')
      print("    작성시간".ljust(10))
      for article in reversed(articles):
          print(f"{article.id}".ljust(8),end='/')
          print(f"     {article.title}".ljust(10),end='/')
          print(f"     {article.content}".ljust(10),end='/')
          print(f"     {article.writeId}".ljust(10),end='/')
          print(f"     {format_date(article.regDate)}")

      print("========================================================")

  else:
    print("지원하지 않은 명령어 입니다.")

print("== CLI 게시판 종료 ==")
